###how many orders did each seller successfully deliver?, How long is the delivery and shipping time? How many orders were late and on-time? What's the percentage of late orders?

In [0]:
output_delivery_performance_path = 's3://ims-ecom-stage/delivery_performance/'

In [0]:
delivery_performance_df = spark.sql("""
SELECT
    s.seller_id,
    s.seller_city,
    s.seller_state,
    COUNT(DISTINCT o.order_id) AS total_orders,
    AVG(o.delivery_days) AS avg_delivery_days,

    AVG(CASE WHEN o.shipping_days < 0 THEN NULL ELSE o.shipping_days END) AS avg_shipping_days,

    COUNT(DISTINCT CASE WHEN o.late_delivery_flag = true THEN o.order_id END) AS late_orders,

    COUNT(DISTINCT CASE WHEN o.late_delivery_flag = false THEN o.order_id END) AS on_time_orders,

    ROUND(100.0 * COUNT(DISTINCT CASE WHEN o.late_delivery_flag = true THEN o.order_id END) / 
    NULLIF(COUNT(DISTINCT CASE WHEN o.late_delivery_flag IS NOT NULL THEN o.order_id END), 0), 2)
    AS late_percentage

FROM ecommerce.staging.orders o
join ecommerce.staging.order_items oi
on o.order_id = oi.order_id
join ecommerce.staging.sellers s
on oi.seller_id = s.seller_id

GROUP BY
    s.seller_id,
    s.seller_city,
    s.seller_state
""")

In [0]:
delivery_performance_df.write \
    .format('delta')  \
    .mode("overwrite") \
    .save(output_delivery_performance_path)

In [0]:
  %sql
CREATE TABLE IF NOT EXISTS ecommerce.staging.delivery_performance
USING delta
LOCATION 's3://ims-ecom-stage/delivery_performance/';